In [1]:
# Install uv for fast package management
!pip install uv

# Install required libraries
!uv pip install torch torchvision torchaudio wandb fiftyone huggingface_hub scikit-learn matplotlib tqdm --system

Using Python 3.11.11 environment at: C:\Users\Philipp\miniconda3
Resolved 134 packages in 1.73s
 Downloaded wandb
Prepared 3 packages in 3.23s
Installed 11 packages in 25.01s
 + gitdb==4.0.12
 + gitpython==3.1.45
 + mpmath==1.3.0
 + protobuf==6.33.2
 + sentry-sdk==2.48.0
 + smmap==5.0.2
 + sympy==1.14.0
 + torch==2.9.1
 + torchaudio==2.9.1
 + torchvision==0.24.1
 + wandb==0.23.1


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import fiftyone as fo
import fiftyone.utils.huggingface as fouh
import wandb
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
!hf auth login

User is already logged in.


In [12]:
!wandb login

wandb: Currently logged in as: hpi-philipp-kolbe (na_mst_2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Load Dataset
Load the dataset from Hugging Face (created in Task 2).

In [3]:
# Configuration
HF_USERNAME = "philippkolbe" 
SUBSET_NAME = "cilp_assessment_subset"
HF_DATASET_REPO = f"{HF_USERNAME}/{SUBSET_NAME}"

# Load dataset from Hugging Face
# This will download the data if not present
print(f"Loading dataset from {HF_DATASET_REPO}...")
try:
    dataset = fouh.load_from_hub(HF_DATASET_REPO)
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Make sure you are logged in to Hugging Face (huggingface-cli login) or have the dataset locally.")

# Print summary
print(dataset)

Loading dataset from philippkolbe/cilp_assessment_subset...


fiftyone.yml:   0%|          | 0.00/107 [00:00<?, ?B/s]

Loading dataset


c:\Users\Philipp\2_uni\wise2526\AHOCV\Applied-Hands-On-Computer-Vision\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Philipp\.cache\huggingface\hub\datasets--philippkolbe--cilp_assessment_subset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Importing samples...
 100% |███████████████| 2250/2250 [123.8ms elapsed, 0s remaining, 18.2K samples/s]    


100%|██████████| 23/23 [05:06<00:00, 13.35s/it]

Dataset loaded successfully.
Name:        philippkolbe/cilp_assessment_subset
Media type:  group
Group slice: rgb
Num groups:  750
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.Metadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    group:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.groups.Group)
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    raw_filepath:     fiftyone.core.fields.StringField


## 2. Prepare Data Loaders
Create a PyTorch Dataset class to handle RGB and LiDAR pairs.

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, fiftyone_dataset, split, transform=None):
        self.transform = transform
        self.samples = []
        
        # Filter by split tag
        view = fiftyone_dataset.match_tags(split)
        
        # Get RGB and LiDAR slices
        rgb_view = view.select_group_slices("rgb")
        lidar_view = view.select_group_slices("lidar")
        
        # Create a map of group_id -> lidar_filepath
        lidar_map = {s.group.id: s.filepath for s in lidar_view}
        
        # Pair them up
        for s in rgb_view:
            if s.group.id in lidar_map:
                self.samples.append({
                    "rgb_path": s.filepath,
                    "lidar_path": lidar_map[s.group.id],
                    "label": 0 if s.ground_truth.label == "cubes" else 1 # cubes=0, spheres=1
                })
                
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # Load RGB (H, W, 4) -> Drop alpha if needed, but let's keep 4 as per assignment
        # Assignment says "RGB (4 channels)" - likely RGBA
        try:
            rgb = plt.imread(item["rgb_path"]) # Returns float 0-1
            if rgb.shape[2] == 3:
                # Pad to 4 channels if RGB
                rgb = np.dstack((rgb, np.ones((rgb.shape[0], rgb.shape[1]))))
        except Exception:
            # Fallback for safety
            rgb = np.zeros((64, 64, 4), dtype=np.float32)

        # Load LiDAR (H, W)
        try:
            lidar = np.load(item["lidar_path"])
            # Normalize LiDAR to 0-1 range roughly for stability
            if lidar.max() - lidar.min() != 0:
                lidar = (lidar - lidar.min()) / (lidar.max() - lidar.min())
        except Exception:
            lidar = np.zeros((64, 64), dtype=np.float32)
            
        # Convert to Tensor
        # RGB: (H, W, C) -> (C, H, W)
        rgb_tensor = torch.from_numpy(rgb).permute(2, 0, 1).float()
        
        # LiDAR: (H, W) -> (1, H, W)
        lidar_tensor = torch.from_numpy(lidar).unsqueeze(0).float()
        
        label_tensor = torch.tensor(item["label"], dtype=torch.float32)
        
        return rgb_tensor, lidar_tensor, label_tensor

# Create Datasets
train_dataset = MultimodalDataset(dataset, "train")
val_dataset = MultimodalDataset(dataset, "val")
test_dataset = MultimodalDataset(dataset, "test")

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 3. Define Models

### 3.1 Late Fusion Architecture
Process modalities separately and concatenate final embeddings.

In [ ]:
class LateFusionModel(nn.Module):
    def __init__(self):
        super(LateFusionModel, self).__init__()
        
        # RGB Encoder
        self.rgb_encoder = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 64 -> 32
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 32 -> 16
            nn.Flatten(),
            nn.Linear(100 * 16 * 16, 100),
            nn.ReLU()
        )
        
        # LiDAR Encoder
        self.lidar_encoder = nn.Sequential(
            nn.Conv2d(1, 50, kernel_size=3, padding=1), # LiDAR has 1 channel
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(100 * 16 * 16, 100),
            nn.ReLU()
        )
        
        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Linear(200, 100), # 100 RGB + 100 LiDAR
            nn.ReLU(),
            nn.Linear(100, 1)
        )
        
    def forward(self, rgb, lidar):
        rgb_emb = self.rgb_encoder(rgb)
        lidar_emb = self.lidar_encoder(lidar)
        
        combined = torch.cat((rgb_emb, lidar_emb), dim=1)
        output = self.classifier(combined)
        return output

### 3.2 Intermediate Fusion Architecture
Combine feature maps at an intermediate layer.

In [ ]:
class IntermediateFusionModel(nn.Module):
    def __init__(self, fusion_type='concat'):
        super(IntermediateFusionModel, self).__init__()
        self.fusion_type = fusion_type
        
        # Early Convolutions (RGB)
        self.rgb_conv = nn.Sequential(
            nn.Conv2d(4, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output: 100 x 16 x 16
        )
        
        # Early Convolutions (LiDAR)
        self.lidar_conv = nn.Sequential(
            nn.Conv2d(1, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output: 100 x 16 x 16
        )
        
        # Determine input channels for shared layers
        if fusion_type == 'concat':
            shared_in_channels = 200
        else: # add or multiply
            shared_in_channels = 100
            
        # Shared Layers
        self.shared_layers = nn.Sequential(
            nn.Conv2d(shared_in_channels, 200, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 16 -> 8
            nn.Flatten(),
            nn.Linear(200 * 8 * 8, 100),
            nn.ReLU(),
            nn.Linear(100, 1)
        )
        
    def forward(self, rgb, lidar):
        x_rgb = self.rgb_conv(rgb)
        x_lidar = self.lidar_conv(lidar)
        
        if self.fusion_type == 'concat':
            combined = torch.cat((x_rgb, x_lidar), dim=1)
        elif self.fusion_type == 'add':
            combined = x_rgb + x_lidar
        elif self.fusion_type == 'multiply':
            combined = x_rgb * x_lidar
        else:
            raise ValueError(f"Unknown fusion type: {self.fusion_type}")
            
        output = self.shared_layers(combined)
        return output

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, config, project_name="cilp-extended-assessment"):
    # Initialize W&B
    wandb.init(project=project_name, config=config, reinit=True)
    
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    best_val_loss = float('inf')
    
    print(f"Starting training for {config['architecture']} ({config['fusion_strategy']})...")
    
    for epoch in range(config["epochs"]):
        # Training
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []
        
        for rgb, lidar, labels in train_loader:
            rgb, lidar, labels = rgb.to(device), lidar.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(rgb, lidar).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy() > 0.5)
            train_targets.extend(labels.cpu().numpy())
            
        avg_train_loss = train_loss / len(train_loader)
        train_acc = accuracy_score(train_targets, train_preds)
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for rgb, lidar, labels in val_loader:
                rgb, lidar, labels = rgb.to(device), lidar.to(device), labels.to(device)
                outputs = model(rgb, lidar).squeeze()
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_preds.extend(torch.sigmoid(outputs).cpu().numpy() > 0.5)
                val_targets.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_targets, val_preds)
        val_f1 = f1_score(val_targets, val_preds)
        
        # Log to W&B
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_acc": train_acc,
            "val_loss": avg_val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        })
        
        print(f"Epoch {epoch+1}/{config['epochs']} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
    # Finish run
    wandb.finish()
    
    return {
        "val_loss": avg_val_loss,
        "val_f1": val_f1,
        "parameters": sum(p.numel() for p in model.parameters())
    }

## 5. Run Experiments

In [ ]:
# Login to W&B
wandb.login()

results = {}

# Common Config
EPOCHS = 10
LR = 1e-3

# 1. Late Fusion
config_late = {
    "architecture": "Late Fusion",
    "fusion_strategy": "late",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_late = LateFusionModel()
results["Late Fusion"] = train_model(model_late, train_loader, val_loader, config_late)

# 2. Intermediate Fusion (Concat)
config_concat = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "concat",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_concat = IntermediateFusionModel(fusion_type='concat')
results["Intermediate (Concat)"] = train_model(model_concat, train_loader, val_loader, config_concat)

# 3. Intermediate Fusion (Add)
config_add = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "add",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_add = IntermediateFusionModel(fusion_type='add')
results["Intermediate (Add)"] = train_model(model_add, train_loader, val_loader, config_add)

# 4. Intermediate Fusion (Multiply)
config_mult = {
    "architecture": "Intermediate Fusion",
    "fusion_strategy": "multiply",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE
}
model_mult = IntermediateFusionModel(fusion_type='multiply')
results["Intermediate (Multiply)"] = train_model(model_mult, train_loader, val_loader, config_mult)

## 6. Comparison Results

| Metric | Late Fusion | Intermediate (Concat) | Intermediate (Add) | Intermediate (Hadamard) |
|---|---:|---:|---:|---:|
| Validation Loss | | | | |
| F1 score | | | | |
| Parameters (count) | | | | |

In [ ]:
print(f"{'Architecture':<25} | {'Val Loss':<10} | {'F1 Score':<10} | {'Params':<10}")
print("-" * 65)
for name, res in results.items():
    print(f"{name:<25} | {res['val_loss']:.4f}     | {res['val_f1']:.4f}     | {res['parameters']}")